## 1. Environment Setup and Library Imports

In this section, we set up the Python environment for the Fuzzy Expert System.
* **Installation:** We install the `fuzzy-expert` library, which provides the core tools for building fuzzy logic controllers and inference systems.
* **Imports:** We import standard data manipulation tools (`pandas`, `numpy`), visualization libraries (`matplotlib`), and machine learning metrics (`sklearn`).
* **Fuzzy Components:** We specifically import classes for defining variables, rules, and the inference engine.

In [8]:
import sys
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from fuzzy_expert.inference import DecompositionalInference
from fuzzy_expert.rule import FuzzyRule
from fuzzy_expert.variable import FuzzyVariable
from fuzzy_expert.mf import MembershipFunction

warnings.filterwarnings('ignore')
print("Libraries imported and environment configured.")

Libraries imported and environment configured.


## 2. Data Loading and Preprocessing (WBCD Dataset)

Here we prepare the Wisconsin Breast Cancer Diagnostic dataset. The goal is to transform raw CSV data into a format suitable for our fuzzy logic system.

**Key Steps:**
1.  **Cleaning:** Removal of whitespace from column headers and standardizing names.
2.  **Encoding:** Mapping the target variable `Diagnosis` to binary values (1 = Malignant, 0 = Benign).
3.  **Feature Selection:** We select five high-impact features (radius, concave points, texture, compactness, area) known to correlate with malignancy.
4.  **Splitting:** The data is split into training (70%) and testing (30%) sets using stratification to maintain class balance.

In [9]:
DATA_SOURCE = "data.csv"
TARGET_COLUMN = "Diagnosis"
SELECTED_FEATURES = [
    'radius_mean',
    'concave_points_mean',
    'texture_mean',
    'compactness_mean',
    'area_mean',
]

def load_and_split_data(filepath, feature_list, target_col):
    """
    Loads dataset, cleans column names, encodes target, and performs stratified split.
    """
    df_raw = pd.read_csv(filepath)
    
    df_raw.columns = df_raw.columns.str.strip()
    
    column_mapping = {
        'diagnosis': target_col,
        'concave points_mean': 'concave_points_mean'
    }
    df_raw.rename(columns=column_mapping, inplace=True)
    
    df_raw[target_col] = df_raw[target_col].map({'M': 1, 'B': 0})
    
    processed_df = df_raw[[target_col] + feature_list].copy()
    
    return train_test_split(
        processed_df,
        test_size=0.3,
        stratify=processed_df[target_col],
        random_state=42
    )

train_df, test_df = load_and_split_data(DATA_SOURCE, SELECTED_FEATURES, TARGET_COLUMN)

print(f"Dataset Columns: {train_df.columns.tolist()}")
print(f"Training Data Dimensions: {train_df.shape}")

Dataset Columns: ['Diagnosis', 'radius_mean', 'concave_points_mean', 'texture_mean', 'compactness_mean', 'area_mean']
Training Data Dimensions: (398, 6)


## 3. Designing Fuzzy Linguistic Variables

This section defines the inputs and outputs for the fuzzy system.

* **Input Variables (Adaptive):** Instead of using arbitrary hard-coded limits, we use a **data-driven approach**. We define the terms `Low`, `Medium`, and `High` based on the statistical quartiles (Q1, Q2, Q3) of the training data. This ensures the fuzzy sets adapt to the actual data distribution.
* **Output Variable (Risk):** We define a custom variable `Risk` ranging from 0 to 1 with two terms: `Benign` and `Malignant`. We deliberately introduce an overlap between 0.3 and 0.7 to handle borderline cases effectively.

In [10]:
fuzzy_vars = {}

def create_quartile_based_variable(data_series):
    """
    Generates a FuzzyVariable with Low, Medium, High terms based on 
    quartiles (Q1, Q2, Q3) of the input data series.
    """
    stats = data_series.astype(float).describe()
    q1, q2, q3 = stats['25%'], stats['50%'], stats['75%']
    min_val, max_val = stats['min'], stats['max']

    if min_val == max_val:
        max_val = min_val + 0.001

    mf_builder = MembershipFunction(n_points=101)

    term_low = mf_builder.trapmf(
        left_feet=min_val, left_peak=min_val, right_peak=q1, right_feet=q2
    )
    term_med = mf_builder.trapmf(
        left_feet=q1, left_peak=q2, right_peak=q2, right_feet=q3
    )
    term_high = mf_builder.trapmf(
        left_feet=q2, left_peak=q3, right_peak=max_val, right_feet=max_val
    )

    return FuzzyVariable(
        universe_range=(min_val, max_val),
        terms={"low": term_low, "medium": term_med, "high": term_high}
    )

for feature in SELECTED_FEATURES:
    fuzzy_vars[feature] = create_quartile_based_variable(train_df[feature])

mf_risk = MembershipFunction(n_points=101)

term_benign = mf_risk.trapmf(
    left_feet=0.0, left_peak=0.0, right_peak=0.3, right_feet=0.7
)

term_malignant = mf_risk.trapmf(
    left_feet=0.3, left_peak=0.7, right_peak=1.0, right_feet=1.0
)

fuzzy_vars["risk"] = FuzzyVariable(
    universe_range=(0.0, 1.0),
    terms={"benign": term_benign, "malignant": term_malignant}
)

print(f"Fuzzy Variables Initialized: {list(fuzzy_vars.keys())}")

Fuzzy Variables Initialized: ['radius_mean', 'concave_points_mean', 'texture_mean', 'compactness_mean', 'area_mean', 'risk']


## 4. Rule Base Construction and Inference Engine

Here we translate expert medical knowledge into fuzzy **IF-THEN** rules.

* **Rule Logic:**
    * **High Priority (Killer Rules):** If specific critical indicators (like `concave_points_mean`) are High, the risk is immediately classified as Malignant.
    * **Safety Rules:** A case is only classified as Benign if multiple features indicate low values simultaneously.
* **Inference Engine:** We configure a **Mamdani-type** inference system using standard operators (Min for AND, Max for OR) to process these rules.

In [11]:
rule_set = []

rule_set.extend([
    FuzzyRule(premise=[("concave_points_mean", "high")], consequence=[("risk", "malignant")]),
    FuzzyRule(premise=[("radius_mean", "high")], consequence=[("risk", "malignant")]),
    FuzzyRule(premise=[("area_mean", "high")], consequence=[("risk", "malignant")]),
    FuzzyRule(
        premise=[("compactness_mean", "high"), ("AND", "texture_mean", "high")],
        consequence=[("risk", "malignant")]
    )
])

rule_set.extend([
    FuzzyRule(
        premise=[("radius_mean", "low"), ("AND", "concave_points_mean", "low")],
        consequence=[("risk", "benign")]
    ),
    FuzzyRule(
        premise=[("texture_mean", "low"), ("AND", "radius_mean", "low")],
        consequence=[("risk", "benign")]
    )
])

# --- Group 3: Intermediate/Mixed Rules ---
rule_set.extend([
    FuzzyRule(
        premise=[("radius_mean", "medium"), ("AND", "concave_points_mean", "high")],
        consequence=[("risk", "malignant")]
    ),
    FuzzyRule(
        premise=[("radius_mean", "medium"), ("AND", "concave_points_mean", "low")],
        consequence=[("risk", "benign")]
    )
])

def build_inference_system(defuzzification_method="cog"):
    """
    Constructs the Mamdani inference engine with specific operators.
    """
    return DecompositionalInference(
        and_operator="min",
        or_operator="max",
        implication_operator="Rc",      # Mamdani min-implication
        composition_operator="max-min",
        production_link="max",
        defuzzification_operator=defuzzification_method
    )

print(f"Total Rules Defined: {len(rule_set)}")

Total Rules Defined: 8


In [12]:
def run_evaluation(f_vars, f_rules, dataset, defuzz_method, decision_threshold):
    """
    Runs the fuzzy inference system on the test set and calculates accuracy.
    """
    engine = build_inference_system(defuzz_method)
    
    actual_labels = dataset[TARGET_COLUMN].values
    predicted_labels = []
    
    for _, record in dataset.iterrows():
        inputs = {f: float(record[f]) for f in SELECTED_FEATURES}
        
        result, _ = engine(variables=f_vars, rules=f_rules, **inputs)
        
        risk_score = result["risk"]
        classification = 1 if risk_score >= decision_threshold else 0
        predicted_labels.append(classification)
        
    return accuracy_score(actual_labels, predicted_labels)

defuzz_strategies = ["cog", "mom"]
threshold_levels = [0.5, 0.6, 0.7]
experiment_log = []

print("Starting Evaluation Loop...")
for method in defuzz_strategies:
    for threshold in threshold_levels:
        current_acc = run_evaluation(
            fuzzy_vars, rule_set, test_df, 
            defuzz_method=method, 
            decision_threshold=threshold
        )
        experiment_log.append({
            "Method": method,
            "Threshold": threshold,
            "Accuracy": current_acc
        })
        print(f"Config: {method.upper()} | Thr: {threshold:.1f} | Accuracy: {current_acc:.3%}")

best_result = max(experiment_log, key=lambda x: x["Accuracy"])
print("\n" + "="*40)
print(f"OPTIMAL CONFIGURATION: Method={best_result['Method']}, "
      f"Threshold={best_result['Threshold']:.1f}, "
      f"Accuracy={best_result['Accuracy']:.3%}")
print("="*40)

Starting Evaluation Loop...
Config: COG | Thr: 0.5 | Accuracy: 85.380%
Config: COG | Thr: 0.6 | Accuracy: 89.474%
Config: COG | Thr: 0.7 | Accuracy: 90.058%
Config: MOM | Thr: 0.5 | Accuracy: 85.380%
Config: MOM | Thr: 0.6 | Accuracy: 85.380%
Config: MOM | Thr: 0.7 | Accuracy: 85.380%

OPTIMAL CONFIGURATION: Method=cog, Threshold=0.7, Accuracy=90.058%
